<a href="https://colab.research.google.com/github/OlhaZahrebelna/Intelligent-Support-Ticket-Router-using-NLP/blob/main/notebook/02_Preprocessing_and_Baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Preprocessing and Classical Baseline Models

This notebook develops and evaluates classical NLP baselines for support-ticket routing.
The workflow is:

1. Load and inspect the cleaned ticket dataset.
2. Create stratified training, validation, and test subsets.
3. Apply text preprocessing inside scikit-learn pipelines to prevent data leakage.
4. Convert text into TF-IDF features.
5. Train and compare Logistic Regression and Linear SVM baselines.
6. Tune the strongest classical model with stratified cross-validation.
7. Evaluate the selected model once on the untouched test set.
8. Summarize model limitations and common sources of classification error.

## Import Libraries

In [1]:
!apt-get update -y
!apt-get install python3.11.9

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
E: Unable to locate package python3.11.9
E: Couldn't find any package by glob 'python3.11.9'
E

In [2]:

import re

# Third-party libraries
import joblib
import matplotlib.pyplot as plt
import pandas as pd
import spacy

# Scikit-learn
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    f1_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

## Load the Dataset

In [3]:
df = pd.read_csv("/content/customer_support_tickets_cleaned.csv")

## Explore the Dataset

In [4]:
df.head()

,text,subject,body,queue,priority,type,language
0,"Account Disruption Dear Customer Support Team,...",Account Disruption,"Dear Customer Support Team,\n\nI am writing to...",Technical Support,high,Incident,en
1,Query About Smart Home System Integration Feat...,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Returns and Exchanges,medium,Request,en
2,Inquiry Regarding Invoice Details Dear Custome...,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",Billing and Payments,low,Request,en
3,Question About Marketing Agency Software Compa...,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Sales and Pre-Sales,medium,Problem,en
4,"Feature Query Dear Customer Support,\n\nI hope...",Feature Query,"Dear Customer Support,\n\nI hope this message ...",Technical Support,high,Request,en


In [5]:
df.shape

(23748, 7)

#### Check missing values

In [6]:
df.isnull().sum()

,0
text,0
subject,2827
body,1
queue,0
priority,0
type,0
language,0


In [7]:
df.duplicated().sum()

np.int64(0)

## Define Features and Target

`X` contains the ticket text used as model input.
`y` contains the queue labels that the model needs to predict.

In [8]:
X = df['text']
y = df['queue']

## Stratified Train–Validation–Test Split

The data is first divided into training and test sets. The training portion is then split again into training and validation subsets for the initial baseline comparison.

Stratification preserves the class distribution across all subsets, which is important because the queue labels are imbalanced. The test set remains untouched until the final model has been selected.

In [9]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.2,
    random_state=42,
    stratify=y_train_full
)

## Text Preprocessing

The custom `TextPreprocessor` class is compatible with scikit-learn pipelines.
This is important because preprocessing must be applied only inside the training pipeline, avoiding data leakage.

The class performs:

- lowercasing;
- URL removal;
- email removal;
- HTML tag removal;
- whitespace normalization;
- lemmatization with spaCy;
- stopword removal;
- removal of numbers and non-alphabetic tokens;
- removal of very short tokens.

In [10]:
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])


class TextPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, min_token_len=2):
        self.min_token_len = min_token_len

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return [self.preprocess(text) for text in X]

    def preprocess(self, text):
        if not isinstance(text, str):
            text = ""

        text = text.lower()

        # remove urls
        text = re.sub(r"http\S+|www\S+", " ", text)

        # remove emails
        text = re.sub(r"\S+@\S+", " ", text)

        # remove html tags
        text = re.sub(r"<.*?>", " ", text)

        # normalize spaces before spaCy
        text = re.sub(r"\s+", " ", text).strip()

        doc = nlp(text)

        tokens = [
            token.lemma_
            for token in doc
            if token.is_alpha
            and not token.is_stop
            and not token.like_num
            and len(token.lemma_) >= self.min_token_len
        ]

        return " ".join(tokens)

### Preprocessing Rationale

This preprocessing design is suitable for a traditional TF-IDF + linear model baseline.
It reduces vocabulary noise while preserving meaningful lexical information.

Important note: the model still depends on words and short phrases. Therefore, the preprocessing should not be too aggressive.
For example, removing all domain-specific words or product names could hurt performance because these tokens may help identify the correct queue.

## TF-IDF

TF-IDF converts cleaned text into sparse numerical vectors.
Terms that are frequent in a specific ticket but not too common across all tickets receive higher weights.

This representation is a strong baseline for classical machine learning models in text classification.

## Baseline Model: Logistic Regression

Logistic Regression is a good first baseline for text classification because it is fast, interpretable, and works well with sparse TF-IDF features.

In [11]:
logreg_pipeline = Pipeline([
    ("preprocessing", TextPreprocessor()),
    ("tfidf", TfidfVectorizer()),
    ("classifier", LogisticRegression(max_iter=1000))
])

In [12]:
logreg_pipeline.fit(X_train, y_train)

y_train_pred_log = logreg_pipeline.predict(X_train)
y_val_pred_log = logreg_pipeline.predict(X_val)

## Logistic Regression Metrics

Macro F1 is especially important here because the dataset is likely imbalanced.
It gives equal weight to each class, so poor performance on minority classes is not hidden by strong performance on large classes.

In [13]:
print("Logistic Regression")
print(classification_report(y_val, y_val_pred_log))

print("Macro F1 Train:",
      f1_score(y_train, y_train_pred_log, average="macro"))
print("Macro F1 Val:",
      f1_score(y_val, y_val_pred_log, average="macro"))

Logistic Regression
                                 precision    recall  f1-score   support

           Billing and Payments       0.83      0.66      0.74       387
               Customer Service       0.33      0.36      0.34       571
                General Inquiry       1.00      0.02      0.04        55
                Human Resources       0.90      0.12      0.21        74
                     IT Support       0.44      0.22      0.29       453
                Product Support       0.38      0.37      0.38       709
          Returns and Exchanges       0.52      0.07      0.13       188
            Sales and Pre-Sales       0.82      0.12      0.21       116
Service Outages and Maintenance       0.76      0.46      0.57       150
              Technical Support       0.44      0.72      0.54      1097

                       accuracy                           0.45      3800
                      macro avg       0.64      0.31      0.35      3800
                   weighted a

## Experiment 2: Logistic Regression with Tuned TF-IDF Features

This experiment modifies both the TF-IDF representation and the classifier settings:

- `ngram_range=(1, 2)`: includes unigrams and bigrams, such as `password reset` or `billing issue`;
- `min_df=5`: removes terms that occur in fewer than five documents;
- `max_df=0.95`: removes extremely common terms;
- `max_features=50_000`: limits the vocabulary size;
- `sublinear_tf=True`: reduces the influence of repeated terms;
- `class_weight="balanced"`: increases the relative importance of minority classes;
- `C=0.1`: applies stronger regularization.


In [14]:
logreg_pipeline2 = Pipeline([
    ("preprocessing", TextPreprocessor()),
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=5,
        max_df=0.95,
        max_features=50_000,
        sublinear_tf=True
    )),
    ("classifier", LogisticRegression(
        max_iter=2_000,
        class_weight="balanced",
        C=0.1
    ))
])

In [15]:
logreg_pipeline2.fit(X_train, y_train)

y_train_pred_log2 = logreg_pipeline2.predict(X_train)
y_val_pred_log2 = logreg_pipeline2.predict(X_val)

## Evaluation of Tuned Logistic Regression


In [16]:
print("Logistic Regression with balanced class weights")
print(classification_report(y_val, y_val_pred_log2))

print("Macro F1 Train:",
      f1_score(y_train, y_train_pred_log2, average="macro"))
print("Macro F1 Val:",
      f1_score(y_val, y_val_pred_log2, average="macro"))

Logistic Regression with balanced class weights
                                 precision    recall  f1-score   support

           Billing and Payments       0.76      0.66      0.71       387
               Customer Service       0.30      0.21      0.25       571
                General Inquiry       0.08      0.47      0.14        55
                Human Resources       0.14      0.43      0.21        74
                     IT Support       0.36      0.28      0.31       453
                Product Support       0.40      0.12      0.19       709
          Returns and Exchanges       0.21      0.26      0.23       188
            Sales and Pre-Sales       0.13      0.58      0.21       116
Service Outages and Maintenance       0.41      0.64      0.50       150
              Technical Support       0.50      0.42      0.46      1097

                       accuracy                           0.35      3800
                      macro avg       0.33      0.41      0.32      3800
 

### Logistic Regression Comparison

The tuned Logistic Regression did **not** improve the validation Macro F1 score. Macro F1 decreased from **0.345** for the default model to **0.320** for the balanced and regularized configuration. Although recall increased for several minority classes, this came with a substantial loss of precision and lower overall accuracy.

This result shows that class balancing and more complex TF-IDF features do not automatically improve performance. The default Logistic Regression therefore remains the stronger of the two Logistic Regression baselines, while Linear SVM should be evaluated as the next classical model.

## Baseline Model 2: Linear SVM

Linear SVM is often very strong for sparse high-dimensional text features.
It tries to find a separating hyperplane with a large margin between classes.

In [17]:
svm_pipeline = Pipeline([
    ("preprocessing", TextPreprocessor()),
    ("tfidf", TfidfVectorizer()),
    ("classifier", LinearSVC())
])

In [18]:
svm_pipeline.fit(X_train, y_train)
y_train_pred_svm = svm_pipeline.predict(X_train)
y_pred_svm = svm_pipeline.predict(X_val)

## Linear SVM Metrics

These metrics are compared with Logistic Regression to identify the stronger baseline model.

In [19]:
print("Linear SVM")
print(classification_report(y_val, y_pred_svm))

print("Macro F1 Train:",
      f1_score(y_train, y_train_pred_svm, average="macro"))
print("Macro F1 Val:",
      f1_score(y_val, y_pred_svm, average="macro"))

Linear SVM
                                 precision    recall  f1-score   support

           Billing and Payments       0.79      0.71      0.75       387
               Customer Service       0.37      0.39      0.38       571
                General Inquiry       0.58      0.20      0.30        55
                Human Resources       0.66      0.28      0.40        74
                     IT Support       0.41      0.28      0.33       453
                Product Support       0.39      0.39      0.39       709
          Returns and Exchanges       0.37      0.18      0.24       188
            Sales and Pre-Sales       0.60      0.28      0.39       116
Service Outages and Maintenance       0.65      0.57      0.60       150
              Technical Support       0.46      0.64      0.54      1097

                       accuracy                           0.47      3800
                      macro avg       0.53      0.39      0.43      3800
                   weighted avg       

## Hyperparameter-Tuned Linear SVM with Cross-Validation

The initial Linear SVM achieved a validation Macro F1 of **0.431**, but its training Macro F1 was substantially higher (**0.684**), indicating a large generalization gap. To improve model selection, the next experiment:

- keeps the test set untouched;
- tunes the regularization parameter `C` with stratified cross-validation;
- tests different TF-IDF vocabulary sizes and minimum document frequencies;
- compares unigram and unigram–bigram representations;
- selects the configuration with the highest mean cross-validation Macro F1.

In [20]:
svm_search_pipeline = Pipeline([
    ("preprocessing", TextPreprocessor()),
    ("tfidf", TfidfVectorizer(
        max_df=0.95,
        sublinear_tf=True
    )),
    ("classifier", LinearSVC(
        class_weight="balanced"
    ))
])

param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [5, 10],
    "tfidf__max_features": [30_000, 50_000],
    "classifier__C": [0.05, 0.1, 0.2]
}

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

grid_search = GridSearchCV(
    estimator=svm_search_pipeline,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=1, # Changed from -1 to 1 to avoid multiprocessing issues with spaCy
    verbose=2,
    return_train_score=True,
    refit=True
)


grid_search.fit(X_train_full, y_train_full)

print("Best parameters:", grid_search.best_params_)
print(f"Best mean CV Macro F1: {grid_search.best_score_:.4f}")

Fitting 3 folds for each of 24 candidates, totalling 72 fits
[CV] END classifier__C=0.05, tfidf__max_features=30000, tfidf__min_df=5, tfidf__ngram_range=(1, 1); total time= 1.4min
[CV] END classifier__C=0.05, tfidf__max_features=30000, tfidf__min_df=5, tfidf__ngram_range=(1, 1); total time= 1.4min
[CV] END classifier__C=0.05, tfidf__max_features=30000, tfidf__min_df=5, tfidf__ngram_range=(1, 1); total time= 1.4min
[CV] END classifier__C=0.05, tfidf__max_features=30000, tfidf__min_df=5, tfidf__ngram_range=(1, 2); total time= 1.4min
[CV] END classifier__C=0.05, tfidf__max_features=30000, tfidf__min_df=5, tfidf__ngram_range=(1, 2); total time= 1.4min
[CV] END classifier__C=0.05, tfidf__max_features=30000, tfidf__min_df=5, tfidf__ngram_range=(1, 2); total time= 1.4min
[CV] END classifier__C=0.05, tfidf__max_features=30000, tfidf__min_df=10, tfidf__ngram_range=(1, 1); total time= 1.4min
[CV] END classifier__C=0.05, tfidf__max_features=30000, tfidf__min_df=10, tfidf__ngram_range=(1, 1); tota

## Cross-Validation Performance and Generalization Gap

For the selected configuration, both mean training and mean validation Macro F1 are reported across the cross-validation folds. The gap helps diagnose overfitting, while mean validation Macro F1 remains the main model-selection metric.

In [21]:
cv_results = pd.DataFrame(grid_search.cv_results_)
best_cv_row = cv_results.loc[grid_search.best_index_]

mean_train_f1 = best_cv_row["mean_train_score"]
mean_val_f1 = best_cv_row["mean_test_score"]
cv_gap = mean_train_f1 - mean_val_f1

print(f"Mean CV Train Macro F1: {mean_train_f1:.4f}")
print(f"Mean CV Validation Macro F1: {mean_val_f1:.4f}")
print(f"Mean CV Generalization Gap: {cv_gap:.4f}")

Mean CV Train Macro F1: 0.8480
Mean CV Validation Macro F1: 0.4806
Mean CV Generalization Gap: 0.3675


## Final Evaluation on the Untouched Test Set

After cross-validation and hyperparameter selection, the best pipeline is evaluated once on the test set. Because the test data was not used for model selection, these results provide the final estimate of out-of-sample performance for the classical baseline.

In [22]:
final_model = grid_search.best_estimator_

y_train_full_pred = final_model.predict(X_train_full)
y_test_pred = final_model.predict(X_test)

train_f1 = f1_score(y_train_full, y_train_full_pred, average="macro")
test_f1 = f1_score(y_test, y_test_pred, average="macro")
final_gap = train_f1 - test_f1

print("Final Regularized Linear SVM")
print(classification_report(y_test, y_test_pred, zero_division=0))
print(f"Macro F1 Train: {train_f1:.4f}")
print(f"Macro F1 Test: {test_f1:.4f}")
print(f"Final Generalization Gap: {final_gap:.4f}")

Final Regularized Linear SVM
                                 precision    recall  f1-score   support

           Billing and Payments       0.79      0.82      0.80       484
               Customer Service       0.51      0.48      0.50       714
                General Inquiry       0.39      0.56      0.46        68
                Human Resources       0.48      0.72      0.58        92
                     IT Support       0.47      0.44      0.45       567
                Product Support       0.55      0.45      0.50       886
          Returns and Exchanges       0.46      0.65      0.54       235
            Sales and Pre-Sales       0.33      0.64      0.44       145
Service Outages and Maintenance       0.52      0.72      0.60       187
              Technical Support       0.64      0.56      0.60      1372

                       accuracy                           0.56      4750
                      macro avg       0.51      0.60      0.55      4750
                   w

### Error Analysis

The final regularized Linear SVM achieved a test Macro F1 score of **0.55**. The model performed best on **Billing and Payments** (F1 = 0.80), where tickets contain relatively distinctive vocabulary. In contrast, the weakest results were observed for **Sales and Pre-Sales** (F1 = 0.44), **IT Support** (F1 = 0.45), and **General Inquiry** (F1 = 0.46).

Most errors are likely caused by semantic overlap between related categories, especially **IT Support**, **Technical Support**, **Product Support**, and **Customer Service**. Short or vague tickets may not contain enough category-specific keywords for a TF-IDF model to distinguish them reliably. Several minority classes also show high recall but low precision, indicating that class balancing causes the model to overpredict them.

The difference between train Macro F1 (**0.86**) and test Macro F1 (**0.55**) shows that overfitting remains present despite regularization. Overall, the model provides a reasonable classical baseline, but a contextual model such as BERT may better capture semantic differences between overlapping ticket categories.


In [23]:
joblib.dump(
    final_model,
    "tfidf_linear_svm_baseline.joblib"
)

['tfidf_linear_svm_baseline.joblib']